# Tutorial 4: Principal Component Analysis (PCA) from Scratch

---

## Learning Objectives

By the end of this tutorial, you will understand:

1. Data representation as a matrix
2. Mean centering
3. Covariance matrix
4. Eigenvalue decomposition
5. Choosing principal components
6. Projecting data
7. PCA for visualization
8. PCA limitations

---

## 1. Data as a Matrix

We represent our dataset as a matrix:

$$X \in \mathbb{R}^{n \times d}$$

- **n** = number of samples
- **d** = number of features

For Iris: $X \in \mathbb{R}^{150 \times 4}$ (150 flowers, 4 features)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Load Iris dataset
iris = load_iris()
X = iris.data
print(f"Shape: {X.shape} (samples, features)")
print(f"Features: {iris.feature_names}")

---

## 2. Mean Centering

PCA requires centered data (mean = 0 for each feature):

$$X_{centered} = X - \mu$$

This ensures covariance measures spread correctly.

In [ ]:
# Compute mean and center data
means = np.mean(X, axis=0)
X_centered = X - means

print("Feature means (original):", np.round(means, 2))
print("Feature means (centered):", np.round(np.mean(X_centered, axis=0), 6))

---

## 3. Covariance Matrix

Covariance tells us how features vary together:

$$Cov(x,y) = \frac{1}{n-1}\sum(x_i - \mu_x)(y_i - \mu_y)$$

The covariance matrix:

$$C = \frac{1}{n-1} X_{centered}^T X_{centered}$$

- **Diagonal**: Variances of each feature
- **Off-diagonal**: Covariances between features

In [ ]:
# Compute covariance matrix
n = X_centered.shape[0]
cov_matrix = (X_centered.T @ X_centered) / (n - 1)

print("Covariance Matrix:")
print(np.round(cov_matrix, 2))

---

## 4. Eigenvalues & Eigenvectors

If $Cv = \lambda v$, then:
- $v$ = eigenvector (direction of a principal component)
- $\lambda$ = eigenvalue (variance in that direction)

**In PCA**: Eigenvectors are the new axes, eigenvalues tell us how much variance each captures.

In [ ]:
# Eigenvalue decomposition
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
eigenvalues = np.real(eigenvalues)
eigenvectors = np.real(eigenvectors)

# Sort in descending order
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]

print("Eigenvalues (sorted):", np.round(eigenvalues, 2))
print("\nEigenvectors (each column):")
print(np.round(eigenvectors, 2))

---

## 5. Explained Variance

How much information does each principal component capture?

$$EVR_i = \frac{\lambda_i}{\sum \lambda}$$

**95% Rule**: Keep components until cumulative variance ≥ 95%

In [ ]:
# Explained variance ratio
total_var = np.sum(eigenvalues)
explained_variance_ratio = eigenvalues / total_var
cumulative_variance = np.cumsum(explained_variance_ratio)

print("Eigenvalue | Variance % | Cumulative %")
print("-" * 40)
for i in range(4):
    print(f"PC{i+1}: {eigenvalues[i]:7.2f} | {explained_variance_ratio[i]*100:6.1f}% | {cumulative_variance[i]*100:6.1f}%")

# Find components for 95%
n_components = np.argmax(cumulative_variance >= 0.95) + 1
print(f"\n→ Need {n_components} components for 95% variance")

In [ ]:
# Visualize explained variance
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.bar(range(1, 5), explained_variance_ratio * 100, color='steelblue', alpha=0.7, label='Individual')
ax.plot(range(1, 5), cumulative_variance * 100, 'ro-', label='Cumulative')
ax.axhline(y=95, color='green', linestyle='--', alpha=0.7, label='95% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('Explained Variance by Principal Components')
ax.legend()
ax.set_xticks(range(1, 5))
plt.tight_layout()
plt.show()

---

## 6. Projection onto Principal Components

Transform data to PCA space:

$$Z = X_{centered} W$$

Where W is the eigenvector matrix (principal components).

In [ ]:
# Project data onto principal components
W = eigenvectors  # Each column is a PC
X_pca = X_centered @ W

print("Original shape:", X.shape)
print("PCA transformed shape:", X_pca.shape)

# Verify: variance should match eigenvalues
pca_variance = np.var(X_pca, axis=0, ddof=1)
print("\nVariance in PCA space:", np.round(pca_variance, 2))
print("Original eigenvalues:", np.round(eigenvalues, 2))

---

## 7. PCA Visualization

Reduce to 2D for plotting:

In [ ]:
# 2D visualization
plt.figure(figsize=(8, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, (color, name) in enumerate(zip(colors, iris.target_names)):
    mask = iris.target == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, alpha=0.7, s=60)

plt.xlabel(f'PC1 ({explained_variance_ratio[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({explained_variance_ratio[1]*100:.1f}%)')
plt.title('PCA Visualization of Iris Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"With 2 components: {cumulative_variance[1]*100:.1f}% variance captured")

---

## 8. IMPORTANT: PCA Limitation

**PCA is UNSUPERVISED** - it does NOT use class labels!

What this means:
- PCA finds directions of **maximum variance**
- It does NOT maximize **class separation**
- The best variance directions may not separate classes well

**For class separation, use supervised methods like LDA!**

In [ ]:
# Demonstrate PCA limitation
print("="*50)
print("PCA LIMITATION DEMONSTRATION")
print("="*50)

print("\nClass centers in PCA space:")
for i, name in enumerate(iris.target_names):
    mask = iris.target == i
    pc1_mean = np.mean(X_pca[mask, 0])
    pc2_mean = np.mean(X_pca[mask, 1])
    print(f"  {name}: PC1={pc1_mean:.2f}, PC2={pc2_mean:.2f}")

print("""
Notice: PCA doesn't guarantee good class separation!
It only maximizes overall variance, not class separation.
Use LDA (Linear Discriminant Analysis) for supervised separation.
""")

---

## Complete PCA Function

Here's everything together:

In [ ]:
def pca_from_scratch(X, variance_threshold=0.95):
    """PCA implementation from scratch."""
    # Step 1: Mean centering
    X_centered = X - np.mean(X, axis=0)
    
    # Step 2: Covariance matrix
    cov = (X_centered.T @ X_centered) / (X.shape[0] - 1)
    
    # Step 3: Eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eig(cov)
    eigenvalues = np.real(eigenvalues)
    eigenvectors = np.real(eigenvectors)
    
    # Step 4: Sort by eigenvalue
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Step 5: Determine components
    total_var = np.sum(eigenvalues)
    exp_ratio = eigenvalues / total_var
    cum_var = np.cumsum(exp_ratio)
    n_components = np.argmax(cum_var >= variance_threshold) + 1
    
    # Step 6: Transform
    W = eigenvectors[:, :n_components]
    X_transformed = X_centered @ W
    
    return X_transformed, W, eigenvalues[:n_components], exp_ratio[:n_components]

# Test it
X_pca_new, components, ev, evr = pca_from_scratch(X)
print(f"Transformed shape: {X_pca_new.shape}")
print(f"Explained variance ratio: {evr * 100}")

---

## Summary

### 8 Key Concepts Covered:

1. ✅ **Data as matrix** - $X \in \mathbb{R}^{n \times d}$
2. ✅ **Mean centering** - $X_{centered} = X - \mu$
3. ✅ **Covariance matrix** - $C = \frac{1}{n-1}X^TX$
4. ✅ **Eigenvalues/eigenvectors** - $Cv = \lambda v$
5. ✅ **Explained variance** - $EVR = \lambda / \sum\lambda$
6. ✅ **Projection** - $Z = X_{centered}W$
7. ✅ **Visualization** - 2D scatter plots
8. ✅ **Limitation** - Unsupervised (no class labels)

---

## Exercises

1. Apply PCA to the wine dataset
2. Compare with sklearn's PCA
3. Visualize in 3D with first 3 PCs
4. Explain: Why might PCA not separate classes?

**End of Tutorial 4**